Sources :

> https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial

> https://debuggercafe.com/transfer-learning-using-efficientnet-pytorch/

>https://pytorch.org/hub/nvidia_deeplearningexamples_efficientnet/

> https://www.kaggle.com/code/shnakazawa/image-classification-with-pytorch-and-efficientnet#Define-the-Model

>https://medium.com/@enrico.randellini/image-classification-resnet-vs-efficientnet-vs-efficientnet-v2-vs-compact-convolutional-c205838bbf49

> https://www.youtube.com/watch?v=qaDe0qQZ5AQ

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile
import os

with zipfile.ZipFile("inference_images.zip", 'r') as zip_ref:
    zip_ref.extractall("inference_images")

ML Start


In [ ]:
import torch
from torchvision.transforms import v2
import cv2
import numpy as np

In [ ]:
transforms = v2.Compose([
    v2.ToImage(),                      # PIL → Tensor benzeri TVTensor
    v2.ToDtype(torch.float32, scale=True),  # [0,255] → [0,1]
    v2.RandomResizedCrop(size=(224, 224), antialias=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


In [ ]:
import os
from torch.utils.data import Dataset
from PIL import Image
class LeavData(Dataset):
    def __init__(self, data_root, transforms=None):
        super().__init__()
        self.data_root = data_root
        self.transforms = transforms

        self.image_paths = []
        self.labels = []

        classes = sorted(os.listdir(data_root))

        for label, class_name in enumerate(classes):
            path = os.path.join(data_root, class_name)


            for img_name in os.listdir(path):
                if img_name.lower().endswith((".jpg")):
                    img_path = os.path.join(path, img_name)
                    self.image_paths.append(img_path)
                    self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, index) :
        img_path = self.image_paths[index]
        label = self.labels[index]

        img = Image.open(img_path).convert("RGB")

        if self.transforms:
            img = self.transforms(img)

        return img, label

In [ ]:
# from sklearn.model_selection import train_test_split
# def split_data(X,y, test_size=0.2, random_state=42):
#   X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.33, random_state=42)
#   return X_train, X_test, y_train, y_test
#

In [ ]:
import torch
from PIL import Image
import numpy as np
import json
import requests
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f'Using {device} for inference')

Using cuda for inference


In [ ]:
from tqdm.auto import tqdm


def train_epoch(model, trainloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_correct = 0

    for i, data in tqdm(enumerate(trainloader), total=len(trainloader)):
        image, label = data
        image = image.to(device)
        label = label.to(device)

        optimizer.zero_grad() # Zero the parameter gradients

        outputs = model(image)
        loss = criterion(outputs, label)

        loss.backward() # Backpropagation
        optimizer.step() # Optimize weights

        running_loss += loss.item() * image.size(0) # Accumulate loss

        # Calculate accuracy
        _, preds = torch.max(outputs.data, 1)
        running_correct += (preds == label).sum().item()

    epoch_loss = running_loss / len(trainloader.dataset)
    epoch_acc = 100. * (running_correct / len(trainloader.dataset))

    return epoch_loss, epoch_acc


def validate_epoch(model, valloader, criterion, device):
    model.eval() # Set the model to evaluation mode
    valid_running_loss = 0.0
    valid_running_correct = 0
    with torch.no_grad():
        for i, data in tqdm(enumerate(valloader), total=len(valloader)):
            image, label = data
            image = image.to(device)
            label = label.to(device)

            outputs = model(image)
            loss = criterion(outputs, label)

            valid_running_loss += loss.item() * image.size(0)
            _, preds = torch.max(outputs.data, 1)
            valid_running_correct += (preds == label).sum().item()

    epoch_loss = valid_running_loss / len(valloader.dataset)
    epoch_acc = 100. * (valid_running_correct / len(valloader.dataset))
    return epoch_loss, epoch_acc

In [ ]:
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Hyperparameters
num_epochs = 58
batch_size = 32
learning_rate = 0.001
num_classes = 8

data_root = "augmented_directory/images"

full_dataset = LeavData(data_root=data_root, transforms=transforms)

# Assuming you want to split data into train and validation.
# The split_data function is defined in exRtBbZsafxk but expects X, y.
# For a Dataset, we typically use random_split from torch.utils.data.

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

model = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_efficientnet_b0', pretrained=True)

# Modify the classifier head for your specific number of classes
# Get the number of features in the last layer
num_ftrs = model.classifier.fc.in_features
model.classifier.fc = nn.Linear(num_ftrs, num_classes)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")

    # Train epoch
    epoch_train_loss, epoch_train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_acc)
    print(f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}%")

    # Validate epoch
    epoch_val_loss, epoch_val_acc = validate_epoch(model, val_loader, criterion, device)
    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_acc)
    print(f"Validation Loss: {epoch_val_loss:.4f}, Validation Acc: {epoch_val_acc:.2f}%")

torch.save(model.state_dict(), "last_model.pth")

Using cache found in /root/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub


Starting training...
Epoch 1/58


  0%|          | 0/259 [00:00<?, ?it/s]

Train Loss: 0.2651, Train Acc: 91.43%


  0%|          | 0/65 [00:00<?, ?it/s]

...

Validation Loss: 0.1225, Validation Acc: 96.76%
Epoch 58/58


  0%|          | 0/259 [00:00<?, ?it/s]

Train Loss: 0.0421, Train Acc: 98.44%


  0%|          | 0/65 [00:00<?, ?it/s]

Validation Loss: 0.0796, Validation Acc: 97.63%
Training finished!


In [ ]:
# Modeli denemek için ai ile yaptım. Buradaki data augment öncesi her resimden ayrı ayrı alınmış ve kategorize edilmiştir.

import os
import torch
from PIL import Image
from torchvision.transforms import v2 # Gerekirse v2'yi tekrar içeri aktar

# Sınıf isimlerini tekrar yükle (data_root'tan gelmeli)
# 'data_root' önceki hücrelerde tanımlanmış olmalıdır (örn: 'augmented_directory/images').
class_names = sorted(os.listdir(data_root))
class_to_idx = {cls_name: i for i, cls_name in enumerate(class_names)}
print(f"Sınıf İsimleri: {class_names}")

# Modeli yükle ve değerlendirme moduna al
# 'model' ve 'device' değişkenlerinin önceki hücrelerden tanımlanmış olduğunu varsayıyoruz.
if 'model' not in locals():
    print("Model bellekte bulunamadı, yeniden yükleniyor...")
    model = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_efficientnet_b0', pretrained=True)
    num_ftrs = model.classifier.fc.in_features
    model.classifier.fc = torch.nn.Linear(num_ftrs, len(class_names))

model.load_state_dict(torch.load('last_model.pth'))
model.eval()
model.to(device)

# Dönüşüm pipeline'ını tekrar tanımla (eğer 'transforms' callable değilse)
if 'transforms' not in locals() or not callable(transforms):
    print("Warning: 'transforms' object not found or not callable. Re-defining default transformations.")
    transforms = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.RandomResizedCrop(size=(224, 224), antialias=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

categorized_images_root = 'categorized_inference_images'
correct_predictions = 0
total_predictions = 0

if not os.path.exists(categorized_images_root):
    print(f"Hata: Kategorize edilmiş resim dizini bulunamadı: {categorized_images_root}")
else:
    print("Kategorize edilmiş resimler üzerinde doğruluk testi başlıyor...")
    for class_folder in os.listdir(categorized_images_root):
        true_class_name = class_folder
        # Sadece bilinen sınıfların klasörlerini işle
        if true_class_name not in class_to_idx:
            print(f"Uyarı: Bilinmeyen sınıf klasörü atlanıyor: {true_class_name}")
            continue

        true_label_idx = class_to_idx[true_class_name]
        folder_path = os.path.join(categorized_images_root, class_folder)

        if os.path.isdir(folder_path):
            for img_name in os.listdir(folder_path):
                img_path = os.path.join(folder_path, img_name)
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    total_predictions += 1
                    try:
                        img_pil = Image.open(img_path).convert("RGB")
                        img_tensor = transforms(img_pil).unsqueeze(0).to(device)

                        with torch.no_grad():
                            outputs = model(img_tensor)
                            _, predicted = torch.max(outputs.data, 1)

                        predicted_class_idx = predicted.item()

                        if predicted_class_idx == true_label_idx:
                            correct_predictions += 1

                    except Exception as e:
                        print(f"'{img_path}' resmi işlenirken hata oluştu: {e}")

    print("Doğruluk testi tamamlandı.")
    if total_predictions > 0:
        accuracy = (correct_predictions / total_predictions) * 100
        print(f"Toplam {total_predictions} resimden {correct_predictions} tanesi doğru tahmin edildi.")
        print(f"Modelin kategorize edilmiş resimler üzerindeki doğruluğu: {accuracy:.2f}%")
    else:
        print("Test edilecek resim bulunamadı.")

Sınıf İsimleri: ['Apple_Black_rot', 'Apple_healthy', 'Apple_rust', 'Apple_scab', 'Grape_Black_rot', 'Grape_Esca', 'Grape_healthy', 'Grape_spot']
Kategorize edilmiş resimler üzerinde doğruluk testi başlıyor...
Doğruluk testi tamamlandı.
Toplam 200 resimden 197 tanesi doğru tahmin edildi.
Modelin kategorize edilmiş resimler üzerindeki doğruluğu: 98.50%
